Imports

In [1]:
# pip install pandas numpy scikit-learn matplotlib seaborn scipy nltk sentence-transformers --quiet


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA

from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42

Data Cleaning

In [ ]:
df = pd.read_csv("AllCorpus.csv")

# Fix 10 rows where category_1 == -1
fixes = {
    101: (5, "none", "Positive"),
    102: (1, "none", "Neutral"),
    103: (4, "none", "Negative"),
    104: (2, "3",    "Positive"),
    105: (1, "3",    "Negative"),
    106: (1, "5",    "Positive"),
    107: (2, "4",    "Neutral"),
    108: (1, "4",    "Neutral"),
    109: (1, "3",    "Positive"),
    110: (2, "3",    "Negative"),
}
for doc_id, (c1, c2, sent) in fixes.items():
    mask = df["doc_id"] == doc_id
    df.loc[mask, "category_1"] = c1
    df.loc[mask, "category_2"] = str(c2)
    df.loc[mask, "sentiment"]  = sent

# Fix doc 396 missing sentiment
df.loc[df["doc_id"] == 396, "sentiment"]  = "Negative"
df.loc[df["doc_id"] == 396, "category_2"] = "none"

# Normalise sentiment spelling/case variants
sentiment_map = {
    "Positive": "Positive", "positive": "Positive",
    "Postive":  "Positive", "Positve":  "Positive",
    "Negative": "Negative", "negative": "Negative",
    "Neutral":  "Neutral",  "neutral":  "Neutral",
    "Nuetral":  "Neutral",  "Neutral.": "Neutral",
    "Neutral ": "Neutral",
}
df["sentiment"] = df["sentiment"].map(sentiment_map)

# Fill missing category_2 with "none", strip whitespace
df["category_2"] = df["category_2"].fillna("none").str.strip()
df["category_1"] = df["category_1"].astype(int)
df = df.reset_index(drop=True)

print("Shape:", df.shape)
print(df.isnull().sum())
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'AllCorpus_-_Corpus.csv'

Vectorization

In [ ]:
bow_vectorizer    = CountVectorizer(stop_words="english")
binary_vectorizer = CountVectorizer(stop_words="english", binary=True)
tfidf_vectorizer  = TfidfVectorizer(stop_words="english")

X_bow    = bow_vectorizer.fit_transform(df["text"])
X_binary = binary_vectorizer.fit_transform(df["text"])
X_tfidf  = tfidf_vectorizer.fit_transform(df["text"])

bow_df    = pd.DataFrame(X_bow.toarray(),    columns=bow_vectorizer.get_feature_names_out())
binary_df = pd.DataFrame(X_binary.toarray(), columns=binary_vectorizer.get_feature_names_out())
tfidf_df  = pd.DataFrame(np.round(X_tfidf.toarray(), 2), columns=tfidf_vectorizer.get_feature_names_out())

display(bow_df.iloc[:5, :12])
display(binary_df.iloc[:5, :12])
display(tfidf_df.iloc[:5, :12])

Cosine Similarity Check

In [ ]:
def similarDocuments(X, vectorizer_name, query_index=0, top_n=2):
    similarities = cosine_similarity(X[query_index], X).ravel()
    neighbors = np.argsort(similarities)[::-1][1:top_n + 1]
    print(f"Query document {query_index}: {df.loc[query_index, 'text']}")
    print(f"Nearest documents using cosine similarity on {vectorizer_name}:")
    for neighbor in neighbors:
        print(f"  {similarities[neighbor]:.2f} | {neighbor}: {df.loc[neighbor, 'text']}")
    print()

similarDocuments(X_tfidf,  "TF-IDF")
similarDocuments(X_binary, "binary presence")

KNN and SVM Helper Functions

In [ ]:
def plot_confusion(y_true, y_pred, title):
    labels = sorted(pd.Series(list(y_true) + list(y_pred)).unique())
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


def safe_train_test_split(X, y):
    return train_test_split(X, y, test_size=0.1)


def evaluate_classifier(model, X, y, title):
    X_train, X_test, y_train, y_test = safe_train_test_split(X, y)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("=" * 80)
    print(title)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    plot_confusion(y_test, y_pred, title)
    return {"title": title, "accuracy": accuracy_score(y_test, y_pred)}

LR and NB Classification

In [ ]:
experiments = []

# Task 1: Category classification
experiments.append(evaluate_classifier(
    LogisticRegression(max_iter=1000), X_bow, df["category_1"],
    "Category Classification by LR on BoW"
))
experiments.append(evaluate_classifier(
    LogisticRegression(max_iter=1000), X_tfidf, df["category_1"],
    "Category Classification by LR on TF-IDF"
))

# Task 2: Sentiment classification
experiments.append(evaluate_classifier(
    MultinomialNB(), X_bow, df["sentiment"],
    "Sentiment Classification by NB on BoW"
))
experiments.append(evaluate_classifier(
    MultinomialNB(), X_tfidf, df["sentiment"],
    "Sentiment Classification by NB on TF-IDF"
))

pd.DataFrame(experiments).sort_values("accuracy", ascending=False)

Single vs. Mixed Classification

In [ ]:
from sklearn.preprocessing import Normalizer

# Binary label: does this doc belong to two categories?
df["is_mixed"] = (df["category_2"] != "none").map({True: "Mixed", False: "Single"})

# L2-normalised BoW = cosine geometry for a linear model
X_bow_cosine = Normalizer().fit_transform(X_bow)

# Task 3: Single vs Mixed — comparing cosine vs binary feature presence
experiments.append(evaluate_classifier(
    LinearSVC(max_iter=2000), X_bow,
    df["is_mixed"], "Single vs Mixed by SVM on BoW (raw counts)"
))
experiments.append(evaluate_classifier(
    LinearSVC(max_iter=2000), X_bow_cosine,
    df["is_mixed"], "Single vs Mixed by SVM on BoW (L2-norm / cosine)"
))
experiments.append(evaluate_classifier(
    LinearSVC(max_iter=2000), X_binary,
    df["is_mixed"], "Single vs Mixed by SVM on Binary Presence"
))
experiments.append(evaluate_classifier(
    LinearSVC(max_iter=2000), X_tfidf,
    df["is_mixed"], "Single vs Mixed by SVM on TF-IDF"
))

pd.DataFrame(experiments).sort_values("accuracy", ascending=False)

KNN Positive vs. Negative Sentiment

In [ ]:
# Subset: only Positive and Negative documents
df_pn = df[df["sentiment"].isin(["Positive", "Negative"])].copy().reset_index(drop=True)

# Refit vectorizers on the subset
bow_pn    = CountVectorizer(stop_words="english")
binary_pn = CountVectorizer(stop_words="english", binary=True)
tfidf_pn  = TfidfVectorizer(stop_words="english")

X_pn_bow    = bow_pn.fit_transform(df_pn["text"])
X_pn_binary = binary_pn.fit_transform(df_pn["text"])
X_pn_tfidf  = tfidf_pn.fit_transform(df_pn["text"])

# L2-normalise for cosine-equivalent KNN
X_pn_tfidf_cos  = Normalizer().fit_transform(X_pn_tfidf)
X_pn_binary_cos = Normalizer().fit_transform(X_pn_binary)

# Task 4: Positive vs Negative — cosine vs binary presence
pn_experiments = []

pn_experiments.append(evaluate_classifier(
    KNeighborsClassifier(n_neighbors=5), X_pn_bow,
    df_pn["sentiment"], "Pos/Neg by KNN on BoW (Euclidean)"
))
pn_experiments.append(evaluate_classifier(
    KNeighborsClassifier(n_neighbors=5), X_pn_tfidf_cos,
    df_pn["sentiment"], "Pos/Neg by KNN on TF-IDF (cosine via L2-norm)"
))
pn_experiments.append(evaluate_classifier(
    KNeighborsClassifier(n_neighbors=5), X_pn_binary_cos,
    df_pn["sentiment"], "Pos/Neg by KNN on Binary Presence (cosine via L2-norm)"
))

pd.DataFrame(pn_experiments).sort_values("accuracy", ascending=False)

Summary Table

In [ ]:
all_experiments = experiments + pn_experiments
pd.DataFrame(all_experiments).sort_values("accuracy", ascending=False)